In [ ]:
%load_ext autoreload
%autoreload 2
import os
import torch
import itertools


from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_raw
from compactreasoningmodels.trace_comparison import Experiment, CKABlock, FlattenBlock, MSEBlock


if 'original_dir' not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")


In [ ]:
dataset = NonogramDataset("traces/nonograms2_5x5.jsonl")
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_raw)

In [ ]:
def make_pair_dataset(dataloader, traces):
    pair_dataset = []

    for X, y, meta in itertools.chain.from_iterable(
        zip(X_batch, y_batch, meta_batch)
        for X_batch, y_batch, meta_batch in dataloader
    ):
        pair = []
        for (trace, version, idx) in traces:
            pair.append(meta["traces"][trace][version][idx]["steps"][:20])
        pair_dataset.append(pair)

    pair_dataset = torch.tensor(
        pair_dataset, dtype=torch.float32, 
        device="cuda" if torch.cuda.is_available() else "cpu"
        ).permute(1, 0, 3, 4, 2)  # (traces, batch, height, width, steps)
    return pair_dataset

In [ ]:
# flatten = FlattenBlock(start_dim=2)
# cka = CKABlock()
# experiment = Experiment([flatten, cka], name="CKA_Comparison")
mse = MSEBlock()
experiment = Experiment([mse], name="MSE_Comparison")

In [ ]:
for solver, sampling_ratio in itertools.product(
    ["mac", "genetic_algorithm_det", 
     "genetic_algorithm_dep", "gradient_descent_global_adam", 
     "gradient_descent_global_sgd", "model_solver"], 
    ["0.8", "0.6"]):
    scores = []
    for idx1, idx2 in itertools.product(range(3), repeat=2):
        traces = [
            (solver, sampling_ratio, idx1),
            (solver, sampling_ratio, idx2)
        ]
        pair_dataset = make_pair_dataset(dataloader, traces)
        result = experiment(pair_dataset)
        scores.append(result)
    adjusted = abs(torch.logit(torch.stack(scores).mean()))
    print(f"{sampling_ratio} {solver:<30}{adjusted.item():.4f}")

In [ ]:
for solver in ["mac", "genetic_algorithm_det", 
     "genetic_algorithm_dep", "gradient_descent_global_adam", 
     "gradient_descent_global_sgd", "model_solver"]:
    for sr1, sr2 in itertools.product(["1.0", "0.8", "0.6"], repeat=2):
        traces = [
            (solver, sr1, 0),
            (solver, sr2, 0)
        ]
        if sr1 == sr2 or float(sr1) < float(sr2):
            continue
        pair_dataset = make_pair_dataset(dataloader, traces)
        result = experiment(pair_dataset)
        adjusted = abs(torch.logit(result))
        print(f"{sr1} {sr2} {solver:<30}{adjusted.item():.4f}")
    print()

In [ ]:
for solver1, solver2 in itertools.product(
    ["mac", "genetic_algorithm_det", 
     "genetic_algorithm_dep", "gradient_descent_global_adam", 
     "gradient_descent_global_sgd", "model_solver"], repeat=2):
    traces = [
        (solver1, "1.0", 0),
        (solver2, "1.0", 0)
    ]
    if solver1 == solver2 or solver1 < solver2:
        continue
    pair_dataset = make_pair_dataset(dataloader, traces)
    result = experiment(pair_dataset)
    adjusted = abs(torch.logit(result))
    print(f"{solver1:<30}{solver2:<30}{adjusted.item():.4f}")

In [ ]:
solvers = ["mac", "genetic_algorithm_det", 
     "genetic_algorithm_dep", "gradient_descent_global_adam", 
     "gradient_descent_global_sgd", "model_solver"]
sampling_ratios = ["1.0", "0.8", "0.6"]
all_configs = itertools.product(solvers, sampling_ratios)
all_pairs = {}
for pair in itertools.combinations(all_configs, r=2):
    traces = [
        (pair[0][0], pair[0][1], 0),
        (pair[1][0], pair[1][1], 0)
    ]
    pair_dataset = make_pair_dataset(dataloader, traces)
    result = experiment(pair_dataset)
    adjusted = abs(torch.logit(result))
    all_pairs[pair] = adjusted.item()


In [ ]:
violations = 0
for pair, value in all_pairs.items():
    for solver in solvers:
        if ((pair[0][0] == solver or pair[1][0] == solver) 
            and value > all_pairs[(solver, "1.0"), (solver, "0.8")]
            and pair[0][0] != pair[1][0]):
            violations += 1
            print("Violation:")
            print(f"  {solver} vs {solver} = {all_pairs[(solver, "1.0"), (solver, "0.8")]:.4f}")
            print(f"  {pair[0]} vs {pair[1]} = {value:.4f}")
